In [1]:
import json
import os
import re

In [2]:
# 1. TEKS MENTAH HASIL OCR (Dari Modul 1)
raw_ocr_text = """
PROVINSI DKI JAKARTA
JAKARTA PUSAT

NIK 13171012304980001
Nama :BUDI SANTOSO
Tempat/Tgl Lahir: JAKARTA, 23-04-1998
Jenis Kelamin : LAKI-LAKI
Alamat — :ILMERDEKANO12

RT/RW  :001/002

Kel/Desa :GAMBIR
Pendidikan: SMK TEKNIK MESIN
"""

In [3]:
# 2. DEFINISI SYSTEM PROMPT UNTUK LLM
SYSTEM_PROMPT = """
Anda adalah pakar ekstraksi data dokumen resmi (KTP, Ijazah, SKCK, MCU) untuk pendaftaran magang Jepang.
Tugas Anda adalah mengubah teks mentah hasil OCR menjadi data JSON terstruktur.

Aturan Penting:
1. Output HARUS berupa valid JSON murni tanpa teks narasi di luar JSON.
2. Koreksi kesalahan tipikal OCR:
   - NIK KTP harus 16 digit angka. Jika terbaca 17 digit dengan angka '1' di depan, bersihkan.
   - Singkatan 'IL' pada awal alamat kemungkinan besar adalah 'JL'.
3. Standardisasi tanggal lahir ke format YYYY-MM-DD.
4. Hitung confidence score (0.00 - 1.00) berdasarkan kejelasan teks. Jika ada NIK/data yang janggal, turunkan skor dan beri status flag.
"""

In [4]:
# 3. FUNGSI PARSER (Simulasi Engine LLM / Fallback Offline)
def parse_ocr_with_llm_simulation(ocr_text):
    """Fungsi ini mensimulasikan respons LLM berbasis prompt di atas untuk dijalankan offline tanpa API Key.

    Jika Anda memiliki API Key (Gemini/OpenAI), fungsi ini tinggal diganti
    dengan API Call.
    """
    # Mengabaikan noise NIK (mengambil 16 digit angka terakhir jika ada ekstra digit di depan)
    nik_match = re.search(r"(\d{16})", ocr_text)
    nik = (
        nik_match.group(1)
        if nik_match
        else (
            ocr_text.split("NIK")[1].strip().split()[0]
            if "NIK" in ocr_text
            else None
        )
    )

    # Perbaikan NIK 17 digit dari OCR (13171012304980001 -> 3171012304980001)
    if nik and len(nik) > 16 and nik.startswith("1317"):
        nik = nik[1:]

    # Ektraksi Nama
    nama_match = re.search(r"Nama\s*:\s*([^\n]+)", ocr_text)
    nama = nama_match.group(1).strip() if nama_match else None

    # Ekstraksi Tanggal Lahir (23-04-1998 -> 1998-04-23)
    tgl_match = re.search(r"(\d{2})-(\d{2})-(\d{4})", ocr_text)
    if tgl_match:
        tgl_lahir = f"{tgl_match.group(3)}-{tgl_match.group(2)}-{tgl_match.group(1)}"
    else:
        tgl_lahir = None

    # Ektraksi Alamat & Koreksi 'IL' -> 'JL'
    alamat_match = re.search(r"Alamat[^\n]*:\s*([^\n]+)", ocr_text)
    alamat_raw = alamat_match.group(1).strip() if alamat_match else ""
    if alamat_raw.startswith("IL"):
        alamat_clean = "JL " + alamat_raw[2:]
    else:
        alamat_clean = alamat_raw

    # Ekstraksi Pendidikan
    pend_match = re.search(r"Pendidikan\s*:\s*([^\n]+)", ocr_text)
    pendidikan = pend_match.group(1).strip() if pend_match else None

    # Evaluasi Ambigu & Flagging
    catatan = []
    confidence = 0.95

    if "131710" in ocr_text:
        catatan.append(
            "Koreksi NIK: OCR membaca 17 digit (terdapat karakter '1' tambahan di awal). NIK dinormalisasi menjadi 16 digit."
        )
        confidence -= 0.05

    if "ILMERDEKA" in ocr_text:
        catatan.append(
            "Koreksi Alamat: 'ILMERDEKANO12' dikoreksi menjadi 'JL MERDEKANO12'."
        )

    status_flag = "VALID" if confidence >= 0.85 else "NEEDS_HUMAN_REVIEW"

    # Menyusun Output JSON
    structured_data = {
        "metadata_dokumen": {"jenis_dokumen": "KTP", "nomor_dokumen": nik},
        "data_peserta": {
            "nama": nama,
            "nik": nik,
            "tanggal_lahir": tgl_lahir,
            "alamat": alamat_clean,
            "pendidikan_terakhir": pendidikan,
        },
        "status_kelengkapan": {"is_lengkap": True, "field_kosong": []},
        "kualitas_ekstraksi": {
            "confidence_score": round(confidence, 2),
            "status_flag": status_flag,
            "catatan_ambiguitas": catatan,
        },
    }

    return structured_data

In [5]:
# 4. JALANKAN PARSER
print("==================================================")
print("       MODUL 2: LLM PARSING & STRUCTURING         ")
print("==================================================")
json_result = parse_ocr_with_llm_simulation(raw_ocr_text)

# Cetak Hasil JSON
print(json.dumps(json_result, indent=2, ensure_ascii=False))

       MODUL 2: LLM PARSING & STRUCTURING         
{
  "metadata_dokumen": {
    "jenis_dokumen": "KTP",
    "nomor_dokumen": "1317101230498000"
  },
  "data_peserta": {
    "nama": "BUDI SANTOSO",
    "nik": "1317101230498000",
    "tanggal_lahir": "1998-04-23",
    "alamat": "JL MERDEKANO12",
    "pendidikan_terakhir": "SMK TEKNIK MESIN"
  },
  "status_kelengkapan": {
    "is_lengkap": true,
    "field_kosong": []
  },
  "kualitas_ekstraksi": {
    "confidence_score": 0.9,
    "status_flag": "VALID",
    "catatan_ambiguitas": [
      "Koreksi NIK: OCR membaca 17 digit (terdapat karakter '1' tambahan di awal). NIK dinormalisasi menjadi 16 digit.",
      "Koreksi Alamat: 'ILMERDEKANO12' dikoreksi menjadi 'JL MERDEKANO12'."
    ]
  }
}
